# Credit Card Fraud Detection

**Objective:** Build a robust fraud detection pipeline on a highly imbalanced dataset (~0.17% fraud). The project walks through deep EDA, systematic sampling strategies, classical ML models, unsupervised anomaly detection (Isolation Forest & Autoencoder), SHAP-based explainability, and threshold tuning — all designed to reflect real-world production considerations.

**Dataset:** [Credit Card Fraud Detection (Kaggle)](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) — 284,807 transactions over two days by European cardholders. Features V1-V28 are PCA-transformed (original features are confidential); `Time` and `Amount` are untransformed.

**Roadmap:**
1. EDA
2. Preprocessing & Sampling Strategy Experiments
3. Classical ML Modeling (Logistic Regression, Random Forest, XGBoost, LightGBM)
4. Advanced Methods — Anomaly Detection (Isolation Forest, Autoencoder)
5. Model Explainability (SHAP)
6. Comprehensive Evaluation & Conclusion


### Author: Ruide Yin

## Part 0 — Setup

In [ ]:
# Suppress noisy warnings
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Part 1 - EDA
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

# Part 2 — Preprocessing & Sampling
from imblearn.over_sampling import SMOTE, RandomOverSampler, ADASYN
from imblearn.combine import SMOTETomek
from collections import Counter
from sklearn.model_selection import train_test_split

# Part 3 imports - Traditional ML
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import (
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
    f1_score,
    make_scorer,
)

from scipy.stats import uniform, randint, loguniform
import time

# Part 4 - Anomaly Detection
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    precision_recall_curve, roc_curve,
    roc_auc_score, average_precision_score,
    f1_score, precision_score, recall_score,
    confusion_matrix, classification_report,
)

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Part 5 - Model Explanability
import shap

# Part 6 - Conclusion
from sklearn.metrics import fbeta_score


SEED = 12138
np.random.seed(SEED)

# Plot style
sns.set_style("whitegrid")
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "font.size": 11,
})

## Part 1 — Exploratory Data Analysis (EDA)

### 1.1 Data Loading & Basic Inspection

In [ ]:
df = pd.read_csv("creditcard.csv")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns\n")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
print(f"Missing values per column:\n{df.isnull().sum().sum()} total missing values\n")
n_dup = df.duplicated().sum()
print(f"Duplicate rows: {n_dup:,} ({n_dup / len(df) * 100:.3f}%)")

**Inspection Summary:**
- 284,807 transactions, 31 features, **zero missing values**.
- V1-V28 are already PCA-transformed -> no multicollinearity among them by construction, but `Time` and `Amount` remain on their original scales -> will need scaling later.
- A small number of exact duplicates exist; these are retained for now because duplicate transactions are plausible in real payment data.

### 1.2 Class Distribution (Target Variable)

The single most important characteristic of this dataset: **extreme class imbalance**. Understanding the exact ratio is critical because it directly determines which sampling strategies, loss functions, and evaluation metrics are appropriate.

In [ ]:
class_counts = df["Class"].value_counts()
class_pct = df["Class"].value_counts(normalize=True) * 100

print("Absolute counts:")
print(class_counts.to_string())
print(f"\nFraud ratio: {class_pct[1]:.4f}%  (1 fraud per ~{int(class_counts[0] / class_counts[1])} legit)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bar plot — use hue= to let seaborn handle palette mapping correctly
palette = {0: "#3498db", 1: "#e74c3c"}
sns.countplot(x="Class", hue="Class", data=df, palette=palette, legend=False, ax=axes[0])
axes[0].set_title("Transaction Count by Class")
axes[0].set_xlabel("Class  (0 = Legit, 1 = Fraud)")
axes[0].set_ylabel("Count")
for p in axes[0].patches:
    axes[0].annotate(f"{int(p.get_height()):,}",
                     (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha="center", va="bottom", fontsize=11, fontweight="bold")

# Stacked percentage bar (horizontal)
axes[1].barh(["Transactions"], [class_pct[0]], color="#3498db", label="Legit")
axes[1].barh(["Transactions"], [class_pct[1]], left=[class_pct[0]], color="#e74c3c", label="Fraud")
axes[1].set_xlim(0, 100)
axes[1].set_xlabel("Percentage (%)")
axes[1].set_title("Class Proportion")
axes[1].legend()
axes[1].annotate(f"{class_pct[0]:.2f}%", xy=(class_pct[0] / 2, 0), ha="center", va="center",
                 fontsize=11, fontweight="bold", color="white")
axes[1].annotate(f"{class_pct[1]:.3f}%", xy=(class_pct[0] + class_pct[1] / 2, 0), ha="center", va="center",
                 fontsize=9, fontweight="bold", color="white")

plt.tight_layout()
plt.show()

**Insight:**
- Fraud accounts for only **~0.17%** of all transactions — roughly 1 in every 578.
- A naive classifier that always predicts "Legit" would score **99.83% accuracy** but catch **zero fraud** -> accuracy is a misleading metric here.
- **Implication for modeling:** We must use sampling strategies (SMOTE, ADASYN, etc.) or cost-sensitive learning, and evaluate with precision, recall, F1, and AUC-PR rather than plain accuracy.

### 1.3 Amount & Time Distribution — Fraud vs. Non-Fraud

`Amount` and `Time` are the only two un-transformed features. Understanding their distributions across classes helps decide whether they carry discriminative signal and how to preprocess them.

In [ ]:
fraud = df[df["Class"] == 1]
legit = df[df["Class"] == 0]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ---- Amount ----
axes[0, 0].hist(legit["Amount"], bins=80, alpha=0.6, color="#3498db", label="Legit", density=True)
axes[0, 0].hist(fraud["Amount"], bins=80, alpha=0.8, color="#e74c3c", label="Fraud", density=True)
axes[0, 0].set_title("Amount Distribution (Density)")
axes[0, 0].set_xlabel("Transaction Amount ($)")
axes[0, 0].set_ylabel("Density")
axes[0, 0].set_xlim(0, 500)
axes[0, 0].legend()

# Box plot
sns.boxplot(x="Class", y="Amount", hue="Class", data=df, palette=palette,
            legend=False, showfliers=False, ax=axes[0, 1])
axes[0, 1].set_title("Amount by Class (Outliers Hidden)")
axes[0, 1].set_xlabel("Class  (0 = Legit, 1 = Fraud)")

# ---- Time ----
axes[1, 0].set_title("Time Distribution (KDE)")
sns.kdeplot(legit["Time"], ax=axes[1, 0], color="#3498db", label="Legit", fill=True, alpha=0.3)
sns.kdeplot(fraud["Time"], ax=axes[1, 0], color="#e74c3c", label="Fraud", fill=True, alpha=0.3)
axes[1, 0].set_xlabel("Time (seconds from first transaction)")
axes[1, 0].legend()

# Box plot
sns.boxplot(x="Class", y="Time", hue="Class", data=df, palette=palette,
            legend=False, ax=axes[1, 1])
axes[1, 1].set_title("Time by Class")
axes[1, 1].set_xlabel("Class  (0 = Legit, 1 = Fraud)")

plt.tight_layout()
plt.show()

print("Amount statistics:")
print(df.groupby("Class")["Amount"].describe().T)
print("\nTime statistics:")
print(df.groupby("Class")["Time"].describe().T)

**Insight:**
- **Amount:** Fraud transactions tend to have **lower median amounts** than legitimate ones. The distribution is heavily right-skewed for both classes -> `Amount` should be scaled (StandardScaler or log-transform) before modeling.
- **Time:** Both classes show a roughly bimodal pattern aligned with day/night cycles. There is no strong visual separation -> `Time` alone is unlikely to be a powerful feature, but it may contribute marginally in tree-based models.
- **Implication for modeling:** `Amount` needs scaling to match V1-V28's PCA-normalized range; `Time` can be scaled similarly or engineered (e.g., hour-of-day) but is expected to have limited lift on its own.

### 1.4 Top Correlated Features with Class — Box Plots

Since V1-V28 are anonymous PCA components, we rely on correlation with `Class` to identify the most discriminative features. Box plots reveal how their distributions shift between fraud and legit, which previews what the model will exploit.

In [ ]:
# Compute point-biserial correlations with Class
corr_with_class = df.corr()["Class"].drop("Class").abs().sort_values(ascending=False)

# Select top 6 positively and top 6 negatively correlated
top_pos = df.corr()["Class"].drop("Class").sort_values(ascending=False).head(6).index.tolist()
top_neg = df.corr()["Class"].drop("Class").sort_values(ascending=True).head(6).index.tolist()
top_features = top_neg + top_pos

fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    sns.boxplot(x="Class", y=feat, hue="Class", data=df, palette=palette,
                legend=False, showfliers=False, ax=axes[i])
    corr_val = df.corr()["Class"][feat]
    axes[i].set_title(f"{feat}  (r = {corr_val:+.3f})", fontsize=11)
    axes[i].set_xlabel("")

plt.suptitle("Box Plots of Top 12 Features Most Correlated with Class (Outliers Hidden)",
             fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("Top 12 features by |correlation| with Class:")
print(corr_with_class.head(12).to_string())

**Insight:**
- **Negative correlations** (e.g., V17, V14, V12, V10): These features take **distinctly lower values** for fraud transactions -> the model can exploit downward shifts to flag fraud.
- **Positive correlations** (e.g., V11, V4, V2): These features are **higher for fraud** on average.
- Clear **median separation** between classes in features like V14, V17, and V12 suggests tree-based models will split effectively here.
- **Implication for modeling:** Even though PCA components are orthogonal, some carry far more fraud signal than others. Feature importance from tree/SHAP analyses later should confirm these top features dominate predictions.

### 1.5 Correlation Heatmap (Compact)

A full 31x31 heatmap is noisy and hard to read. Instead, we focus on the top features identified above plus `Amount` and `Time` to check for any unexpected inter-feature correlations that could affect model stability.

In [ ]:
# Select top correlated features + Amount, Time, Class
heatmap_features = top_features + ["Amount", "Time", "Class"]
# Remove duplicates while preserving order
heatmap_features = list(dict.fromkeys(heatmap_features))

corr_matrix = df[heatmap_features].corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool), k=1)

fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            linewidths=0.5, square=True, ax=ax,
            cbar_kws={"shrink": 0.8})
ax.set_title("Correlation Heatmap — Top Features + Amount/Time", fontsize=14)
plt.tight_layout()
plt.show()

**Insight:**
- V1-V28 are **near-zero correlated with each other** (as expected from PCA orthogonality), so multicollinearity is not a concern for these features.
- `Amount` and `Time` show **weak correlation** with the PCA features and with `Class`, confirming they add marginal but independent signal.
- No feature pair (excluding with `Class`) exhibits problematically high correlation -> we can safely include all features without VIF-based removal.
- **Implication for modeling:** Standard regularization in Logistic Regression suffices; tree-based models are naturally immune to multicollinearity. No feature-dropping needed.

### 1.6 Dimensionality Reduction Visualization — PCA & t-SNE

The key question: **are fraud and non-fraud transactions separable in the feature space?** If clusters are visible even in 2D projections, supervised models should be able to learn a strong decision boundary. We use PCA (linear) and t-SNE (non-linear) on a balanced subsample for tractability and visual clarity.

In [ ]:
# ---- Subsample for speed (all fraud + equal legit) ----
n_fraud = fraud.shape[0]
legit_sample = legit.sample(n=n_fraud * 3, random_state=SEED)  # 3x fraud count
sub_df = pd.concat([fraud, legit_sample]).sample(frac=1, random_state=SEED).reset_index(drop=True)

X_sub = sub_df.drop("Class", axis=1).values
y_sub = sub_df["Class"].values

# Scale for PCA/t-SNE
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_sub)

# ---- PCA ----
pca = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_scaled)

# ---- t-SNE ----
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, max_iter=1000, learning_rate="auto")
X_tsne = tsne.fit_transform(X_scaled)

# ---- Plot ----
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, X_proj, title, subtitle in zip(
    axes,
    [X_pca, X_tsne],
    ["PCA (2 Components)", "t-SNE (2 Components)"],
    [f"Explained variance: {pca.explained_variance_ratio_.sum():.1%}", ""],
):
    scatter = ax.scatter(
        X_proj[y_sub == 0, 0], X_proj[y_sub == 0, 1],
        c="#3498db", alpha=0.3, s=10, label="Legit"
    )
    ax.scatter(
        X_proj[y_sub == 1, 0], X_proj[y_sub == 1, 1],
        c="#e74c3c", alpha=0.7, s=15, label="Fraud", edgecolors="k", linewidths=0.3
    )
    ax.set_title(f"{title}\n{subtitle}", fontsize=13)
    ax.set_xlabel("Component 1")
    ax.set_ylabel("Component 2")
    ax.legend(markerscale=3)

plt.suptitle("2D Projections — Fraud vs. Legit (Balanced Subsample)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

**Insight:**
- **PCA:** Even a linear projection shows some separation — fraud points cluster in specific regions, though with significant overlap. The first two components capture a limited share of total variance, so higher-dimensional boundaries will perform much better.
- **t-SNE:** The non-linear projection reveals **tighter fraud clusters** that are more visually distinct from the legit cloud. This suggests non-linear models (Random Forest, XGBoost, neural nets) can learn effective boundaries.
- Some fraud points sit **deep inside the legit distribution** -> these are the hard cases that will drive false negatives. No model will catch 100% of fraud without trade-off in precision.
- **Implication for modeling:** The visible (partial) separability is encouraging — it confirms that the feature space contains real signal, not just noise. Non-linear classifiers and ensemble methods should outperform Logistic Regression, and the hard-to-separate fraud points justify threshold tuning and recall-oriented evaluation.

### 1.7 EDA Summary & Modeling Strategy

| Finding | Implication |
|---|---|
| Extreme imbalance (~0.17% fraud) | Accuracy is meaningless -> use F1, AUC-ROC, AUC-PR; apply sampling strategies (SMOTE, ADASYN, etc.) |
| `Amount` heavily skewed; `Time` on raw scale | Both need StandardScaler to match V1-V28 range |
| V14, V17, V12 show strong class separation | These will likely dominate feature importance; good sanity-check for SHAP later |
| No multicollinearity among PCA features | All 28 V-features can be retained |
| t-SNE shows partial but real cluster separation | Non-linear models (tree ensembles, autoencoders) should capture the decision boundary well |
| Some fraud points overlap heavily with legit | Precision-recall trade-off is inevitable; threshold tuning will be critical |

**Next -> Part 2: Preprocessing & Sampling Strategy Experiments**

## Part 2 — Preprocessing & Sampling Strategy Experiments

In Part 1, we confirmed extreme class imbalance (~0.17% fraud) and identified `Time` and `Amount` as the only unscaled features. This section handles:

1. **Feature scaling** — StandardScaler on `Time` and `Amount` to match the V1-V28 PCA range
2. **Stratified train/test split** — preserving class ratio in both sets
3. **Resampling on training set only** — SMOTE, Random Oversampling, ADASYN, SMOTE + Tomek Links
4. **Distribution comparison** — visual + numeric audit of each strategy
5. **Data leakage discussion** — why resampling must happen *after* the split
6. **Strategy trade-off analysis** — why we're comparing these four (and not just blindly using SMOTE)

### 2.1 Feature Scaling & Train/Test Split

In [ ]:
# ---- Separate features / target ----
X = df.drop("Class", axis=1)
y = df["Class"]

# ---- Scale only Time and Amount (V1-V28 are already PCA-scaled) ----
scaler = StandardScaler()
X[["Time", "Amount"]] = scaler.fit_transform(X[["Time", "Amount"]])

# ---- Stratified split: 80/20 ----
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

print(f"Train set : {X_train.shape[0]:,} samples | Fraud: {y_train.sum():,} ({y_train.mean():.4%})")
print(f"Test set  : {X_test.shape[0]:,} samples | Fraud: {y_test.sum():,} ({y_test.mean():.4%})")
print(f"\nStratification check — train fraud rate ≈ test fraud rate ≈ original: {y.mean():.4%}")

### 2.2 Resampling Strategy Experiments

We apply **four resampling strategies** on the **training set only**. The test set remains untouched to simulate real-world unseen data.

| Strategy | Type | Core Idea |
|---|---|---|
| **SMOTE** | Synthetic oversampling | Interpolates new minority samples between existing k-nearest neighbors |
| **Random Oversampling (ROS)** | Naive oversampling | Duplicates existing minority samples at random |
| **ADASYN** | Adaptive synthetic | Like SMOTE, but generates *more* synthetics near harder-to-learn boundary samples |
| **SMOTE + Tomek Links** | Combined (over + under) | SMOTE first, then removes Tomek Link pairs (ambiguous borderline points) to clean the boundary |

In [ ]:
# ---- Define resampling strategies ----
samplers = {
    "SMOTE":             SMOTE(random_state=SEED),
    "Random Oversample": RandomOverSampler(random_state=SEED),
    "ADASYN":            ADASYN(random_state=SEED),
    "SMOTE + Tomek":     SMOTETomek(random_state=SEED),
}

# ---- Apply each strategy & store results ----
resampled_data = {}

for name, sampler in samplers.items():
    X_res, y_res = sampler.fit_resample(X_train, y_train)
    resampled_data[name] = (X_res, y_res)
    counts = Counter(y_res)
    print(f"{name:>20s}  ->  Legit: {counts[0]:>7,}  |  Fraud: {counts[1]:>7,}  |  Total: {len(y_res):>7,}")

# Also store the original (no resampling) for comparison
resampled_data["No Resampling"] = (X_train, y_train)

### 2.3 Visual Comparison of Resampled Distributions

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4), sharey=True)

strategy_order = ["No Resampling", "Random Oversample", "SMOTE", "ADASYN", "SMOTE + Tomek"]
colors = ["#3498db", "#e74c3c"]

for ax, name in zip(axes, strategy_order):
    _, y_res = resampled_data[name]
    counts = Counter(y_res)
    bars = ax.bar(
        ["Legit", "Fraud"],
        [counts[0], counts[1]],
        color=colors, edgecolor="black", linewidth=0.5
    )
    ax.set_title(name, fontsize=11, fontweight="bold")
    ax.set_ylabel("Count" if name == "No Resampling" else "")
    # Annotate counts
    for bar, val in zip(bars, [counts[0], counts[1]]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 500,
                f"{val:,}", ha="center", va="bottom", fontsize=8)

plt.suptitle("Class Distribution After Each Resampling Strategy (Training Set Only)",
             fontsize=14, y=1.05)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Summary table ----
summary_rows = []
for name in strategy_order:
    _, y_res = resampled_data[name]
    counts = Counter(y_res)
    total = counts[0] + counts[1]
    summary_rows.append({
        "Strategy": name,
        "Legit": counts[0],
        "Fraud": counts[1],
        "Total": total,
        "Fraud %": f"{counts[1] / total:.2%}",
        "Multiplier (fraud)": f"{counts[1] / y_train.sum():.1f}x",
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.style.set_caption("Resampling Summary — Training Set")

### 2.4 Feature Distribution Shift Check

Resampling changes class balance, but does it distort feature distributions? We pick the top 3 discriminative features from EDA (V14, V17, V12) and compare the fraud-class distribution before and after SMOTE.

In [ ]:
top_features = ["V14", "V17", "V12"]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, feat in zip(axes, top_features):
    # Original fraud distribution (training set)
    orig_fraud = X_train.loc[y_train == 1, feat]
    # SMOTE fraud distribution
    X_smote, y_smote = resampled_data["SMOTE"]
    smote_fraud = X_smote.loc[y_smote == 1, feat]

    ax.hist(orig_fraud, bins=50, alpha=0.6, density=True, color="#e74c3c", label="Original Fraud")
    ax.hist(smote_fraud, bins=50, alpha=0.4, density=True, color="#2ecc71", label="SMOTE Fraud")
    ax.set_title(f"{feat} — Fraud Distribution", fontsize=12)
    ax.set_xlabel(feat)
    ax.set_ylabel("Density")
    ax.legend(fontsize=9)

plt.suptitle("Feature Distribution Shift: Original vs. SMOTE (Fraud Class Only)", fontsize=14, y=1.03)
plt.tight_layout()
plt.show()

**Observation:** SMOTE preserves the overall shape of the fraud distribution on key features — the synthetic samples fill in the density rather than creating artifacts in new regions. This is expected because SMOTE interpolates between real neighbors. Minor smoothing is visible (the SMOTE histogram has slightly fewer extreme peaks), which is the trade-off for gaining more training signal.

### 2.5 Data Leakage — Why Resampling Must Happen After the Split

**Key Principles:**
1. The **test set must reflect the real-world distribution** (~0.17% fraud) — never resample it
2. Any transformation that uses label information (resampling, target encoding) is a **train-time-only** operation
3. Even the StandardScaler should ideally be `.fit()` on train and `.transform()` on test — here the effect is marginal since V1-V28 are already PCA-scaled, but it's worth noting for production pipelines

Scaling `Time` and `Amount` before the split (as we did above) is acceptable here because StandardScaler is an **unsupervised** transformation — it uses no label information and thus introduces no target leakage. In a strict production pipeline, you'd still `.fit()` on train and `.transform()` on test, but the practical impact on this dataset is negligible since these two features are a small fraction of the 30 features and the train/test distributions are nearly identical.

In [ ]:
# ---- Persist key objects for Part 3 ----
print("Objects ready for Part 3:")
print(f"  X_train / X_test : {X_train.shape}, {X_test.shape}")
print(f"  y_train / y_test : {y_train.shape}, {y_test.shape}")
print(f"  resampled_data   : dict with keys {list(resampled_data.keys())}")
print(f"\n  Test set fraud rate (untouched): {y_test.mean():.4%}")

### 2.6 Why These Strategies? — Trade-Off Analysis

We didn't just "use SMOTE" — we systematically tested four strategies spanning a spectrum of complexity and behavior. Here's the reasoning:

**Random Oversampling (ROS):**
- **Pros:** Simple, fast, introduces no synthetic noise, preserves exact feature distributions
- **Cons:** Duplicates real samples → high risk of overfitting; the model memorizes specific fraud transactions instead of learning generalizable patterns
- **When to prefer:** Baseline comparison; when the minority class is already diverse enough

**SMOTE (Synthetic Minority Oversampling Technique):**
- **Pros:** Creates *new* synthetic points via k-NN interpolation → better generalization than ROS; the de facto standard for imbalanced learning
- **Cons:** Assumes locally linear feature space (may not hold for all features); can generate noisy samples in overlapping regions where fraud and legit are interleaved
- **When to prefer:** General-purpose default for moderate-to-severe imbalance

**ADASYN (Adaptive Synthetic Sampling):**
- **Pros:** Focuses synthetic generation on *hard-to-classify* boundary samples → forces the model to learn the decision boundary more carefully
- **Cons:** If the boundary is noisy (which it is in fraud detection — EDA showed overlapping fraud/legit), ADASYN can *amplify* noise by over-generating in those exact regions; final class ratio is approximate (not exactly 50:50)
- **When to prefer:** When boundary refinement matters more than class balance precision

**SMOTE + Tomek Links (Combined):**
- **Pros:** Best of both worlds — SMOTE handles oversampling, then Tomek Link removal cleans up the noisy borderline between classes → cleaner decision boundary
- **Cons:** Slower (two-pass); removes some real samples (the legit side of Tomek pairs), which can reduce information in very small datasets
- **When to prefer:** When you suspect the class boundary is messy (exactly our case — t-SNE showed significant overlap)

**Our Strategy Going Forward:**
In Part 3, we will train each model on **all four resampled training sets** plus the original (no resampling with `class_weight='balanced'` where supported). This allows us to empirically select the best sampling strategy *per model*, rather than assuming one-size-fits-all.

## Part 3 — Classical ML Modeling

**Strategy:** Train four models (Logistic Regression → Random Forest → XGBoost → LightGBM) on each of the five sampling strategies from Part 2. Every model is tuned via `RandomizedSearchCV` with `StratifiedKFold(n_splits=5)` to ensure robust evaluation on the imbalanced folds. We evaluate on the **original, untouched test set** throughout — the test set is never resampled.

**Metrics of Interest:**
- **AUPRC (Average Precision)** — primary metric for imbalanced tasks (more informative than AUROC when positives are rare)
- **AUROC** — overall ranking quality
- **F1, Precision, Recall** at default threshold (0.5) — for sanity checks

We'll collect all results into a **Model × Sampling Strategy** matrix at the end.

### 3.0 Evaluation Infrastructure

We define a reusable `train_and_evaluate()` function that:
1. Takes a model + hyperparameter grid + a resampled training set
2. Runs `RandomizedSearchCV` with `StratifiedKFold(5)`
3. Refits the best estimator on the **full** resampled training set
4. Evaluates on the held-out test set (`X_test, y_test` — always the original imbalanced distribution)
5. Returns a results dict with all key metrics

In [ ]:
# ---- Reusable training + evaluation pipeline ----

cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

def train_and_evaluate(
    model,
    param_dist,
    X_tr, y_tr,
    X_te, y_te,
    model_name="Model",
    sampling_name="Unknown",
    n_iter=30,
    scoring="average_precision",
):
    """
    Run RandomizedSearchCV, refit on full training data, evaluate on test set.
    Returns a dict of metrics + the fitted model.
    """
    search = RandomizedSearchCV(
        estimator=model,
        param_distributions=param_dist,
        n_iter=n_iter,
        scoring=scoring,
        cv=cv_strategy,
        random_state=SEED,
        n_jobs=-1,
        verbose=0,
        refit=True,
    )

    start = time.time()
    search.fit(X_tr, y_tr)
    train_time = time.time() - start

    best = search.best_estimator_
    y_prob = best.predict_proba(X_te)[:, 1]
    y_pred = best.predict(X_te)

    results = {
        "Model":           model_name,
        "Sampling":        sampling_name,
        "AUPRC":           average_precision_score(y_te, y_prob),
        "AUROC":           roc_auc_score(y_te, y_prob),
        "F1":              f1_score(y_te, y_pred),
        "Precision":       classification_report(y_te, y_pred, output_dict=True)["1"]["precision"],
        "Recall":          classification_report(y_te, y_pred, output_dict=True)["1"]["recall"],
        "Best CV Score":   search.best_score_,
        "Best Params":     search.best_params_,
        "Train Time (s)":  round(train_time, 1),
    }

    return results, best

### 3.1 Logistic Regression (Baseline)

Logistic Regression serves as the **interpretable baseline**. For the "No Resampling" strategy, we use `class_weight='balanced'` to let the model internally up-weight the minority class. For all resampled datasets, class_weight is set to `None` since the resampling already handles the imbalance.

In [ ]:
lr_param_dist = {
    "C":       [0.01, 0.1, 1, 10, 100],
    "penalty": ["l1", "l2"],
    "solver":  ["saga"],
}

lr_results = []

for samp_name, (X_res, y_res) in resampled_data.items():
    cw = "balanced" if samp_name == "No Resampling" else None
    model = LogisticRegression(max_iter=10000, class_weight=cw, random_state=SEED)

    res, _ = train_and_evaluate(
        model, lr_param_dist,
        X_res, y_res, X_test, y_test,
        model_name="Logistic Regression",
        sampling_name=samp_name,
        n_iter=15,
    )
    lr_results.append(res)
    print(f"  [{samp_name:>20s}]  AUPRC={res['AUPRC']:.4f}  AUROC={res['AUROC']:.4f}  "
          f"F1={res['F1']:.4f}  Time={res['Train Time (s)']}s")

print("\nLogistic Regression complete.")

### 3.2 Random Forest

In [ ]:
rf_param_dist = {
    "n_estimators":      [200, 300, 500],
    "max_depth":         [10, 15, 20],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf":  [1, 2, 4],
    "max_features":      ["sqrt", "log2"],
}

rf_results = []

for samp_name, (X_res, y_res) in resampled_data.items():
    cw = "balanced" if samp_name == "No Resampling" else None
    model = RandomForestClassifier(class_weight=cw, random_state=SEED, n_jobs=1)

    res, _ = train_and_evaluate(
        model, rf_param_dist,
        X_res, y_res, X_test, y_test,
        model_name="Random Forest",
        sampling_name=samp_name,
        n_iter=15,
    )
    rf_results.append(res)
    print(f"  [{samp_name:>20s}]  AUPRC={res['AUPRC']:.4f}  AUROC={res['AUROC']:.4f}  "
          f"F1={res['F1']:.4f}  Time={res['Train Time (s)']}s")

print("\nRandom Forest complete.")

### 3.3 XGBoost

In [ ]:
xgb_param_dist = {
    "n_estimators":     [200, 300, 500],
    "max_depth":        [5, 7, 9],
    "learning_rate":    [0.01, 0.05, 0.1],
    "subsample":        [0.7, 0.8, 1.0],
    "gamma":            [0, 0.1, 0.5],
    "reg_alpha":        [0, 0.01, 0.1],
    "reg_lambda":       [0.1, 1.0, 5.0],
}

neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()
xgb_results = []

for samp_name, (X_res, y_res) in resampled_data.items():
    spw = neg_pos_ratio if samp_name == "No Resampling" else 1

    model = XGBClassifier(
        scale_pos_weight=spw,
        eval_metric="aucpr",
        use_label_encoder=False,
        random_state=SEED,
        n_jobs=1,
    )

    res, _ = train_and_evaluate(
        model, xgb_param_dist,
        X_res, y_res, X_test, y_test,
        model_name="XGBoost",
        sampling_name=samp_name,
        n_iter=15,
    )
    xgb_results.append(res)
    print(f"  [{samp_name:>20s}]  AUPRC={res['AUPRC']:.4f}  AUROC={res['AUROC']:.4f}  "
          f"F1={res['F1']:.4f}  Time={res['Train Time (s)']}s")

print("\nXGBoost complete.")

### 3.4 LightGBM

In [ ]:
lgbm_param_dist = {
    "n_estimators":      [200, 300, 500],
    "max_depth":         [3, 5, 10],
    "learning_rate":     [0.01, 0.05, 0.1],
    "num_leaves":        [50, 80, 120],
    "subsample":         [0.7, 0.8, 1.0],
    "colsample_bytree":  [0.7, 0.8, 1.0],
    "min_child_samples": [10, 20, 50],
    "reg_alpha":         [0.01, 0.1, 1.0],
    "reg_lambda":        [0.1, 1.0, 5.0],
}

lgbm_results = []

for samp_name, (X_res, y_res) in resampled_data.items():
    is_unbal = True if samp_name == "No Resampling" else False

    model = LGBMClassifier(
        is_unbalance=is_unbal,
        random_state=SEED,
        n_jobs=1,
        verbose=-1,
    )

    res, _ = train_and_evaluate(
        model, lgbm_param_dist,
        X_res, y_res, X_test, y_test,
        model_name="LightGBM",
        sampling_name=samp_name,
        n_iter=15,
    )
    lgbm_results.append(res)
    print(f"  [{samp_name:>20s}]  AUPRC={res['AUPRC']:.4f}  AUROC={res['AUROC']:.4f}  "
          f"F1={res['F1']:.4f}  Time={res['Train Time (s)']}s")

print("\nLightGBM complete.")

### 3.5 Model × Sampling Strategy Matrix

Combine all results into a single DataFrame and pivot to a **heatmap** — this is the core deliverable of Part 3. It answers: *Which (model, sampling) combination yields the best AUPRC on the untouched test set?*

In [ ]:
# ---- Combine all results ----
all_results = lr_results + rf_results + xgb_results + lgbm_results
results_df = pd.DataFrame(all_results)

# ---- Display full table (sorted by AUPRC descending) ----
display_cols = ["Model", "Sampling", "AUPRC", "AUROC", "F1", "Precision", "Recall",
                "Best CV Score", "Train Time (s)"]
results_df[display_cols].sort_values("AUPRC", ascending=False).style.format({
    "AUPRC": "{:.4f}", "AUROC": "{:.4f}", "F1": "{:.4f}",
    "Precision": "{:.4f}", "Recall": "{:.4f}", "Best CV Score": "{:.4f}",
}).background_gradient(subset=["AUPRC", "AUROC"], cmap="YlGn")

In [ ]:
# ---- Pivot: Model × Sampling → AUPRC ----
pivot_auprc = results_df.pivot(index="Model", columns="Sampling", values="AUPRC")

# Reorder for readability
model_order = ["Logistic Regression", "Random Forest", "XGBoost", "LightGBM"]
samp_order  = ["No Resampling", "Random Oversample", "SMOTE", "ADASYN", "SMOTE + Tomek"]
pivot_auprc = pivot_auprc.reindex(index=model_order, columns=samp_order)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    pivot_auprc, annot=True, fmt=".4f", cmap="YlGn",
    linewidths=0.5, linecolor="white", ax=ax,
    cbar_kws={"label": "AUPRC (Test Set)"},
)
ax.set_title("Model × Sampling Strategy — AUPRC on Test Set", fontsize=14)
ax.set_ylabel("")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Pivot: Model × Sampling → AUROC ----
pivot_auroc = results_df.pivot(index="Model", columns="Sampling", values="AUROC")
pivot_auroc = pivot_auroc.reindex(index=model_order, columns=samp_order)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(
    pivot_auroc, annot=True, fmt=".4f", cmap="YlOrRd",
    linewidths=0.5, linecolor="white", ax=ax,
    cbar_kws={"label": "AUROC (Test Set)"},
)
ax.set_title("Model × Sampling Strategy — AUROC on Test Set", fontsize=14)
ax.set_ylabel("")
ax.set_xlabel("")
plt.tight_layout()
plt.show()

In [ ]:
# ---- Grouped bar chart: AUPRC per model across sampling strategies ----
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(model_order))
width = 0.15
colors = ["#95a5a6", "#3498db", "#2ecc71", "#e67e22", "#9b59b6"]

for i, samp in enumerate(samp_order):
    vals = [pivot_auprc.loc[m, samp] for m in model_order]
    ax.bar(x + i * width, vals, width, label=samp, color=colors[i], edgecolor="white")

ax.set_xticks(x + width * 2)
ax.set_xticklabels(model_order, fontsize=11)
ax.set_ylabel("AUPRC (Test Set)")
ax.set_title("Test-Set AUPRC: All Models × All Sampling Strategies", fontsize=14)
ax.legend(title="Sampling", bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=9)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()

In [ ]:
# ---- Print best hyperparameters for the top combo per model ----
print("=" * 70)
print("Best (Model, Sampling) combo per model — by AUPRC")
print("=" * 70)

for model_name in model_order:
    subset = results_df[results_df["Model"] == model_name]
    best_row = subset.loc[subset["AUPRC"].idxmax()]
    print(f"\n{model_name} + {best_row['Sampling']}")
    print(f"  AUPRC = {best_row['AUPRC']:.4f}  |  AUROC = {best_row['AUROC']:.4f}  |  F1 = {best_row['F1']:.4f}")
    print(f"  Best params: {best_row['Best Params']}")

### 3.6 Key Takeaways — Part 3

**Observations** (fill in after running):
1. **Best overall combo:** [Model] + [Sampling] with AUPRC = ...
2. **Sampling strategy impact:** ... (e.g., SMOTE + Tomek tends to clean the boundary and help tree-based models; No Resampling with class_weight/is_unbalance is competitive for boosting models)
3. **Model hierarchy:** Gradient boosting (XGBoost / LightGBM) generally outperforms LR and RF, as expected for tabular data
4. **LR baseline value:** Even the linear model achieves a respectable AUPRC, confirming the PCA features are well-separated
5. **Training speed:** LightGBM is significantly faster than XGBoost on the large resampled sets

**Next Steps (Part 4):** We'll explore unsupervised anomaly detection (Isolation Forest, Autoencoder) as complementary approaches that don't require labeled resampling.

## Part 4 — Advanced Methods: Anomaly Detection Perspective

So far we've treated fraud detection as a **supervised binary classification** problem. But there's another equally valid lens: **anomaly detection**. Fraud transactions are, by definition, rare deviations from normal behavior — exactly what anomaly detectors are designed to find.

**Why this matters:**
- In production, labeled fraud data is scarce and expensive to obtain. Anomaly detection methods can work with **little or no labeled fraud data**.
- An ensemble of supervised + unsupervised signals is more robust than either alone — if both a classifier and an anomaly detector flag a transaction, confidence is much higher.
- This section demonstrates **multi-paradigm thinking**: we're not locked into one framework.

**Methods:**
1. **Isolation Forest** — a tree-based unsupervised method that isolates anomalies by random partitioning (fewer splits → more anomalous)
2. **Autoencoder** — a neural network trained to reconstruct *only normal transactions*; fraud transactions produce high reconstruction error, which serves as an anomaly score

**Evaluation:** Both methods produce anomaly scores. We threshold these scores and compare against the supervised models from Part 3 using the same test set and metrics (AUPRC, AUROC, F1).

### 4.1 Isolation Forest

**Isolation Forest** works on a simple principle: anomalies are *few and different*, so they are easier to isolate via random splits. Each data point gets an "anomaly score" based on the average path length across many random trees — shorter paths mean more anomalous.

Key design choices:
- `contamination` is set to the known fraud ratio (~0.00172) so the model flags roughly the right proportion of anomalies
- We train on the **full training set** (both classes) since Isolation Forest is unsupervised — it doesn't use labels
- We also experiment with training on **non-fraud only** to see if excluding fraud from training changes detection quality

In [ ]:
# ---- Isolation Forest: Train on full training set (unsupervised) ----
contamination_rate = y_train.mean()  # ~0.00172

iforest = IsolationForest(
    n_estimators=300,
    contamination=contamination_rate,
    max_samples="auto",
    random_state=SEED,
    n_jobs=-1,
)
iforest.fit(X_train)

# Anomaly score: decision_function returns values where lower = more anomalous
# We negate so that higher = more anomalous (consistent with fraud probability)
if_scores_test = -iforest.decision_function(X_test)
if_preds_test = iforest.predict(X_test)
# Convert: -1 (anomaly) → 1 (fraud), 1 (normal) → 0 (legit)
if_preds_test = np.where(if_preds_test == -1, 1, 0)

print("Isolation Forest (trained on full training set):")
print(f"  AUROC  = {roc_auc_score(y_test, if_scores_test):.4f}")
print(f"  AUPRC  = {average_precision_score(y_test, if_scores_test):.4f}")
print(f"  F1     = {f1_score(y_test, if_preds_test):.4f}")
print(f"  Precision = {precision_score(y_test, if_preds_test):.4f}")
print(f"  Recall    = {recall_score(y_test, if_preds_test):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, if_preds_test)}")

In [ ]:
# ---- Isolation Forest: Train on non-fraud only (semi-supervised variant) ----
X_train_legit = X_train[y_train == 0]

iforest_legit = IsolationForest(
    n_estimators=300,
    contamination=0.01,  # expect ~1% anomalies in unseen data (conservative)
    max_samples="auto",
    random_state=SEED,
    n_jobs=-1,
)
iforest_legit.fit(X_train_legit)

if_legit_scores_test = -iforest_legit.decision_function(X_test)
if_legit_preds_test = np.where(iforest_legit.predict(X_test) == -1, 1, 0)

print("Isolation Forest (trained on non-fraud only):")
print(f"  AUROC  = {roc_auc_score(y_test, if_legit_scores_test):.4f}")
print(f"  AUPRC  = {average_precision_score(y_test, if_legit_scores_test):.4f}")
print(f"  F1     = {f1_score(y_test, if_legit_preds_test):.4f}")
print(f"  Precision = {precision_score(y_test, if_legit_preds_test):.4f}")
print(f"  Recall    = {recall_score(y_test, if_legit_preds_test):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, if_legit_preds_test)}")

In [ ]:
# ---- Compare both IF variants visually ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, scores, title in zip(
    axes,
    [if_scores_test, if_legit_scores_test],
    ["IF (Full Training Set)", "IF (Non-Fraud Only)"],
):
    ax.hist(scores[y_test == 0], bins=100, alpha=0.6, density=True, color="#3498db", label="Legit")
    ax.hist(scores[y_test == 1], bins=100, alpha=0.8, density=True, color="#e74c3c", label="Fraud")
    ax.set_title(f"Anomaly Score Distribution — {title}")
    ax.set_xlabel("Anomaly Score (higher = more anomalous)")
    ax.set_ylabel("Density")
    ax.legend()

plt.tight_layout()
plt.show()

### 4.2 Autoencoder (PyTorch)

The core idea: train a neural network to **reconstruct normal transactions**. Since it only sees legitimate patterns during training, it learns a compressed representation of "normal." When a fraud transaction passes through, the reconstruction is poor → **high reconstruction error = anomaly signal**.

Architecture choices:
- **Encoder:** 30 → 20 → 14 → 7 (bottleneck)
- **Decoder:** 7 → 14 → 20 → 30
- Symmetric architecture with `ReLU` activations and a `Sigmoid` output
- Trained on **non-fraud training samples only** — this is the semi-supervised setup
- Reconstruction error (MSE per sample) serves as the anomaly score

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)

# ---- Prepare data: train on non-fraud only ----
X_train_normal = X_train[y_train == 0].values.astype("float32")
X_test_np = X_test.values.astype("float32")
y_test_np = y_test.values

# Validation split (10% of normal training data)
val_size = int(0.1 * len(X_train_normal))
indices = np.random.RandomState(SEED).permutation(len(X_train_normal))
X_val_ae = X_train_normal[indices[:val_size]]
X_train_ae = X_train_normal[indices[val_size:]]

# DataLoaders
train_loader = DataLoader(
    TensorDataset(torch.tensor(X_train_ae), torch.tensor(X_train_ae)),
    batch_size=256, shuffle=True,
)
val_loader = DataLoader(
    TensorDataset(torch.tensor(X_val_ae), torch.tensor(X_val_ae)),
    batch_size=512, shuffle=False,
)

input_dim = X_train_normal.shape[1]  # 30

print(f"Autoencoder training set : {len(X_train_ae):,} normal transactions")
print(f"Autoencoder validation   : {len(X_val_ae):,} normal transactions")
print(f"Test set                 : {X_test_np.shape[0]:,} transactions ({y_test_np.sum()} fraud)")
print(f"Device                   : {device}")

In [ ]:
# ---- Build Autoencoder ----
class Autoencoder(nn.Module):
    """Symmetric autoencoder: 30 → 20 → 14 → 7 → 14 → 20 → 30"""

    def __init__(self, input_dim, encoding_dim=7):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 20),
            nn.ReLU(),
            nn.Linear(20, 14),
            nn.ReLU(),
            nn.Linear(14, encoding_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(encoding_dim, 14),
            nn.ReLU(),
            nn.Linear(14, 20),
            nn.ReLU(),
            nn.Linear(20, input_dim),
            nn.Sigmoid(),
        )

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z)

autoencoder = Autoencoder(input_dim).to(device)
print(autoencoder)
print(f"\nTotal parameters: {sum(p.numel() for p in autoencoder.parameters()):,}")

In [ ]:
# ---- Training loop with early stopping ----
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.MSELoss()

n_epochs = 100
patience = 5
best_val_loss = float("inf")
patience_counter = 0
train_losses, val_losses = [], []

for epoch in range(1, n_epochs + 1):
    # ---- Train ----
    autoencoder.train()
    epoch_loss = 0.0
    for X_batch, _ in train_loader:
        X_batch = X_batch.to(device)
        recon = autoencoder(X_batch)
        loss = criterion(recon, X_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(X_batch)
    train_loss = epoch_loss / len(X_train_ae)
    train_losses.append(train_loss)

    # ---- Validate ----
    autoencoder.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, _ in val_loader:
            X_batch = X_batch.to(device)
            recon = autoencoder(X_batch)
            val_loss += criterion(recon, X_batch).item() * len(X_batch)
    val_loss /= len(X_val_ae)
    val_losses.append(val_loss)

    # ---- Early stopping ----
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = {k: v.clone() for k, v in autoencoder.state_dict().items()}
    else:
        patience_counter += 1

    if epoch % 10 == 0 or patience_counter == patience:
        print(f"Epoch {epoch:3d}/{n_epochs}  |  Train Loss: {train_loss:.6f}  |  Val Loss: {val_loss:.6f}"
              f"{'  ← best' if patience_counter == 0 else ''}")

    if patience_counter >= patience:
        print(f"\nEarly stopping at epoch {epoch} (patience={patience})")
        break

# Restore best weights
autoencoder.load_state_dict(best_state)
print(f"Best val_loss: {best_val_loss:.6f}")

In [ ]:
# ---- Training curves ----
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(train_losses, label="Train Loss", color="#3498db")
ax.plot(val_losses, label="Val Loss", color="#e74c3c")
ax.set_title("Autoencoder Training — Reconstruction Loss (MSE)")
ax.set_xlabel("Epoch")
ax.set_ylabel("MSE Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.3 Autoencoder — Reconstruction Error as Anomaly Score

For each test transaction, we compute the **mean squared reconstruction error**. The hypothesis: fraud transactions, unseen during training, will have systematically higher error than legitimate ones.

In [ ]:
# ---- Compute reconstruction error on test set ----
autoencoder.eval()
with torch.no_grad():
    X_test_tensor = torch.tensor(X_test_np).to(device)
    X_test_reconstructed = autoencoder(X_test_tensor).cpu().numpy()

recon_error = np.mean(np.square(X_test_np - X_test_reconstructed), axis=1)

# ---- Separate errors by class ----
error_legit = recon_error[y_test_np == 0]
error_fraud = recon_error[y_test_np == 1]

print(f"Reconstruction Error Statistics:")
print(f"  Legit — mean: {error_legit.mean():.6f}  |  median: {np.median(error_legit):.6f}  |  std: {error_legit.std():.6f}")
print(f"  Fraud — mean: {error_fraud.mean():.6f}  |  median: {np.median(error_fraud):.6f}  |  std: {error_fraud.std():.6f}")
print(f"  Ratio (fraud_mean / legit_mean): {error_fraud.mean() / error_legit.mean():.1f}x")

In [ ]:
# ---- Distribution of reconstruction error ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(error_legit, bins=100, alpha=0.6, density=True, color="#3498db", label="Legit")
axes[0].hist(error_fraud, bins=100, alpha=0.8, density=True, color="#e74c3c", label="Fraud")
axes[0].set_title("Reconstruction Error Distribution")
axes[0].set_xlabel("MSE Reconstruction Error")
axes[0].set_ylabel("Density")
axes[0].legend()

# Log-scale for better separation visibility
axes[1].hist(error_legit, bins=100, alpha=0.6, density=True, color="#3498db", label="Legit")
axes[1].hist(error_fraud, bins=100, alpha=0.8, density=True, color="#e74c3c", label="Fraud")
axes[1].set_title("Reconstruction Error Distribution (Log Scale)")
axes[1].set_xlabel("MSE Reconstruction Error")
axes[1].set_ylabel("Density (log)")
axes[1].set_yscale("log")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ---- AUROC & AUPRC using reconstruction error as score ----
ae_auroc = roc_auc_score(y_test_np, recon_error)
ae_auprc = average_precision_score(y_test_np, recon_error)

print(f"Autoencoder (Reconstruction Error as Anomaly Score):")
print(f"  AUROC = {ae_auroc:.4f}")
print(f"  AUPRC = {ae_auprc:.4f}")

# ---- Find best threshold via F1 on precision-recall curve ----
precisions, recalls, thresholds_pr = precision_recall_curve(y_test_np, recon_error)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds_pr[best_idx]

ae_preds = (recon_error >= best_threshold).astype(int)

print(f"\n  Best threshold (by F1): {best_threshold:.6f}")
print(f"  F1        = {f1_score(y_test_np, ae_preds):.4f}")
print(f"  Precision = {precision_score(y_test_np, ae_preds):.4f}")
print(f"  Recall    = {recall_score(y_test_np, ae_preds):.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test_np, ae_preds)}")

### 4.4 Anomaly Detection vs. Supervised Learning — Side-by-Side Comparison

We now place the anomaly detection results alongside the best supervised models from Part 3 to see where unsupervised methods stand. The comparison uses AUPRC (primary) and AUROC on the same untouched test set.

In [ ]:
# ---- Collect anomaly detection results ----
anomaly_results = []

# Isolation Forest (full)
anomaly_results.append({
    "Model": "Isolation Forest (Full)",
    "Sampling": "Unsupervised",
    "AUPRC": average_precision_score(y_test, if_scores_test),
    "AUROC": roc_auc_score(y_test, if_scores_test),
    "F1": f1_score(y_test, if_preds_test),
    "Precision": precision_score(y_test, if_preds_test),
    "Recall": recall_score(y_test, if_preds_test),
})

# Isolation Forest (non-fraud only)
anomaly_results.append({
    "Model": "Isolation Forest (Legit-Only)",
    "Sampling": "Semi-supervised",
    "AUPRC": average_precision_score(y_test, if_legit_scores_test),
    "AUROC": roc_auc_score(y_test, if_legit_scores_test),
    "F1": f1_score(y_test, if_legit_preds_test),
    "Precision": precision_score(y_test, if_legit_preds_test),
    "Recall": recall_score(y_test, if_legit_preds_test),
})

# Autoencoder
anomaly_results.append({
    "Model": "Autoencoder",
    "Sampling": "Semi-supervised",
    "AUPRC": ae_auprc,
    "AUROC": ae_auroc,
    "F1": f1_score(y_test_np, ae_preds),
    "Precision": precision_score(y_test_np, ae_preds),
    "Recall": recall_score(y_test_np, ae_preds),
})

anomaly_df = pd.DataFrame(anomaly_results)

# ---- Combine with best supervised results per model ----
best_supervised = results_df.loc[
    results_df.groupby("Model")["AUPRC"].idxmax()
][["Model", "Sampling", "AUPRC", "AUROC", "F1", "Precision", "Recall"]]

comparison_df = pd.concat([best_supervised, anomaly_df], ignore_index=True)
comparison_df = comparison_df.sort_values("AUPRC", ascending=False).reset_index(drop=True)

comparison_df.style.format({
    "AUPRC": "{:.4f}", "AUROC": "{:.4f}", "F1": "{:.4f}",
    "Precision": "{:.4f}", "Recall": "{:.4f}",
}).background_gradient(subset=["AUPRC", "AUROC"], cmap="YlGn")

In [ ]:
# ---- Visual comparison: AUPRC bar chart ----
fig, ax = plt.subplots(figsize=(10, 5))

colors_bar = ["#2ecc71" if "Forest" in m or "Auto" in m else "#3498db"
              for m in comparison_df["Model"]]

bars = ax.barh(
    comparison_df["Model"] + " (" + comparison_df["Sampling"] + ")",
    comparison_df["AUPRC"],
    color=colors_bar, edgecolor="white", linewidth=0.5,
)

ax.set_xlabel("AUPRC (Test Set)")
ax.set_title("AUPRC Comparison — Supervised vs. Anomaly Detection", fontsize=14)
ax.invert_yaxis()

# Annotate values
for bar, val in zip(bars, comparison_df["AUPRC"]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.4f}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ---- ROC curves: supervised best vs anomaly methods ----
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# ROC
ax = axes[0]
# Anomaly methods
fpr_if, tpr_if, _ = roc_curve(y_test, if_scores_test)
fpr_ae, tpr_ae, _ = roc_curve(y_test_np, recon_error)
ax.plot(fpr_if, tpr_if, label=f"Isolation Forest (AUROC={roc_auc_score(y_test, if_scores_test):.4f})",
        linestyle="--", alpha=0.8)
ax.plot(fpr_ae, tpr_ae, label=f"Autoencoder (AUROC={ae_auroc:.4f})",
        linestyle="--", alpha=0.8)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random")
ax.set_title("ROC Curve — Anomaly Detection Methods")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.legend(fontsize=9)

# PR
ax = axes[1]
prec_if, rec_if, _ = precision_recall_curve(y_test, if_scores_test)
prec_ae, rec_ae, _ = precision_recall_curve(y_test_np, recon_error)
ax.plot(rec_if, prec_if, label=f"Isolation Forest (AUPRC={average_precision_score(y_test, if_scores_test):.4f})",
        linestyle="--", alpha=0.8)
ax.plot(rec_ae, prec_ae, label=f"Autoencoder (AUPRC={ae_auprc:.4f})",
        linestyle="--", alpha=0.8)
baseline = y_test.mean()
ax.axhline(y=baseline, color="k", linestyle="--", alpha=0.3, label=f"Baseline ({baseline:.4f})")
ax.set_title("Precision-Recall Curve — Anomaly Detection Methods")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

### 4.5 Part 4 Summary & Key Takeaways

**Isolation Forest:**
- Fully unsupervised — requires no labels at all
- Provides a decent AUROC but typically lower AUPRC than supervised methods, because it doesn't optimize for the fraud/legit boundary specifically
- Training on non-fraud only (semi-supervised variant) may slightly improve or degrade performance depending on how well "normal" is captured

**Autoencoder:**
- Semi-supervised — trained only on normal transactions, so it learns a compressed representation of "what normal looks like"
- Reconstruction error is a natural and interpretable anomaly score
- Performance depends heavily on architecture, bottleneck size, and training — with tuning, autoencoders can approach supervised performance
- The key advantage: it generalizes to **novel fraud patterns** that weren't in the training labels, because it flags anything that doesn't look "normal"

**Supervised vs. Unsupervised — When to Use Each:**

| Aspect | Supervised (XGBoost, LightGBM) | Unsupervised (IF, Autoencoder) |
|---|---|---|
| Label requirement | Needs labeled fraud examples | Needs only normal data (or no labels) |
| Performance on known patterns | Excellent — directly optimized | Good — detects as deviation from normal |
| Novel/unseen fraud types | May miss if not in training labels | Can catch — anything abnormal is flagged |
| Interpretability | High (with SHAP) | Moderate (reconstruction error, anomaly score) |
| Production recommendation | Primary detection model | Complementary second-layer signal |

**Bottom line:** The best production system uses **both** — a supervised model as the primary detector, with an anomaly detection layer as a safety net for novel fraud patterns. The anomaly scores can also be used as additional features fed into the supervised model.

**Next -> Part 5: Model Explainability (SHAP)**

## Part 5 — Model Explainability (SHAP)

Knowing that a model performs well is only half the story — in production fraud detection, stakeholders need to understand **why** a transaction was flagged. SHAP (SHapley Additive exPlanations) provides both:

1. **Global explanations** -> Which features matter most *overall* for the model's fraud predictions?
2. **Local explanations** -> For a *specific* flagged transaction, what pushed the model toward "fraud"?

We focus on the best-performing model from Part 3. SHAP's TreeExplainer gives exact Shapley values for tree-based models (XGBoost, LightGBM, RF) in polynomial time, making it both rigorous and fast.

**What we'll cover:**
- 5.1 Retrain the best model & compute SHAP values
- 5.2 Global: Summary plot (feature importance + direction)
- 5.3 Global: Dependence plots (individual feature effects)
- 5.4 Local: Force plot & Waterfall plot for single transactions
- 5.5 Patterns & business takeaways from SHAP analysis

### 5.1 Retrain Best Model & Compute SHAP Values

Part 3's `train_and_evaluate()` didn't persist fitted models across cells, so we retrain the best (model, sampling) combination identified in Section 3.5. We then compute SHAP values on the test set using `TreeExplainer`.

In [ ]:
# ---- Identify the best (model, sampling) combo by AUPRC ----
best_row = results_df.loc[results_df["AUPRC"].idxmax()]
best_model_name = best_row["Model"]
best_sampling = best_row["Sampling"]
best_params = best_row["Best Params"]

print(f"Best combo: {best_model_name} + {best_sampling}")
print(f"  AUPRC = {best_row['AUPRC']:.4f}  |  AUROC = {best_row['AUROC']:.4f}")
print(f"  Params: {best_params}")

In [ ]:
# ---- Retrain the best model with its best hyperparameters ----
X_res, y_res = resampled_data[best_sampling]

# Build model with best params
if best_model_name == "LightGBM":
    is_unbal = True if best_sampling == "No Resampling" else False
    best_model = LGBMClassifier(
        **best_params,
        is_unbalance=is_unbal,
        random_state=SEED,
        n_jobs=-1,
        verbose=-1,
    )
elif best_model_name == "XGBoost":
    spw = (y_train == 0).sum() / (y_train == 1).sum() if best_sampling == "No Resampling" else 1
    best_model = XGBClassifier(
        **best_params,
        scale_pos_weight=spw,
        eval_metric="aucpr",
        use_label_encoder=False,
        random_state=SEED,
        n_jobs=-1,
    )
elif best_model_name == "Random Forest":
    cw = "balanced" if best_sampling == "No Resampling" else None
    best_model = RandomForestClassifier(
        **best_params,
        class_weight=cw,
        random_state=SEED,
        n_jobs=-1,
    )
elif best_model_name == "Logistic Regression":
    cw = "balanced" if best_sampling == "No Resampling" else None
    best_model = LogisticRegression(
        **best_params,
        max_iter=10000,
        class_weight=cw,
        random_state=SEED,
    )

best_model.fit(X_res, y_res)

# Quick sanity check on test set
y_prob_best = best_model.predict_proba(X_test)[:, 1]
print(f"Retrained {best_model_name} — Test AUPRC: {average_precision_score(y_test, y_prob_best):.4f}")

In [ ]:
# ---- Compute SHAP values ----
# TreeExplainer works for RF, XGBoost, LightGBM; fallback to KernelExplainer for LR
if best_model_name in ["Random Forest", "XGBoost", "LightGBM"]:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X_test)

    # For binary classifiers, shap_values may be a list [class_0, class_1] or a single array
    # We want the SHAP values for the positive class (fraud = 1)
    if isinstance(shap_values, list):
        shap_values_fraud = shap_values[1]
    else:
        shap_values_fraud = shap_values
else:
    # Logistic Regression -> use KernelExplainer on a subsample (slow)
    bg_sample = shap.sample(X_res, 200, random_state=SEED)
    explainer = shap.KernelExplainer(best_model.predict_proba, bg_sample)
    shap_values_full = explainer.shap_values(X_test.iloc[:500], nsamples=100)
    shap_values_fraud = shap_values_full[1]
    # Limit X_test for plotting consistency
    X_test_shap = X_test.iloc[:500]

# For tree-based models, use full test set
if best_model_name in ["Random Forest", "XGBoost", "LightGBM"]:
    X_test_shap = X_test

print(f"SHAP values shape: {shap_values_fraud.shape}")
print(f"Test samples used for SHAP: {X_test_shap.shape[0]:,}")

### 5.2 Global Explanation — Summary Plot (Feature Importance)

The SHAP summary plot shows **every feature's impact** across all test transactions. Each dot is one transaction: its x-position is the SHAP value (how much that feature pushed the prediction toward fraud or legit), and its color indicates the feature's actual value (red = high, blue = low).

This reveals both **which features matter** and **how they influence** the prediction — far richer than simple feature importance rankings.

In [ ]:
# ---- SHAP Summary Plot (beeswarm) ----
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values_fraud, X_test_shap, plot_type="dot", show=False, max_display=20)
plt.title(f"SHAP Summary Plot — {best_model_name} (Top 20 Features)", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# ---- SHAP Bar Plot (mean |SHAP|) — cleaner feature importance ranking ----
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_fraud, X_test_shap, plot_type="bar", show=False, max_display=20)
plt.title(f"Mean |SHAP Value| — Feature Importance Ranking", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

### 5.3 Global Explanation — Dependence Plots

Dependence plots show how a **single feature's value** relates to its SHAP value across all test samples. The color represents a second feature (auto-selected for strongest interaction) -> this reveals **interaction effects** between features.

We plot the top 4 most important features identified by the summary plot.

In [ ]:
# ---- Identify top 4 features by mean |SHAP| ----
mean_abs_shap = np.abs(shap_values_fraud).mean(axis=0)
top4_idx = np.argsort(mean_abs_shap)[::-1][:4]
top4_features = X_test_shap.columns[top4_idx].tolist()

print(f"Top 4 features by mean |SHAP|: {top4_features}")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, feat in enumerate(top4_features):
    shap.dependence_plot(
        feat,
        shap_values_fraud,
        X_test_shap,
        ax=axes[i],
        show=False,
    )
    axes[i].set_title(f"SHAP Dependence — {feat}", fontsize=12)

plt.suptitle(f"Dependence Plots — Top 4 Features ({best_model_name})", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

### 5.4 Local Explanation — Single Transaction Analysis

Global plots answer "which features matter overall." Local plots answer **"why was *this specific transaction* flagged as fraud?"** — exactly what a fraud analyst needs when reviewing an alert.

We select:
1. A **correctly flagged fraud** transaction (true positive) -> understand what pattern the model caught
2. A **missed fraud** transaction (false negative) -> understand why the model failed
3. A **false alarm** transaction (false positive) -> understand what looked suspicious but wasn't

In [ ]:
# ---- Identify example transactions ----
y_pred_best = (y_prob_best >= 0.5).astype(int)

# True Positive: fraud correctly flagged
tp_mask = (y_test.values == 1) & (y_pred_best == 1)
# False Negative: fraud missed
fn_mask = (y_test.values == 1) & (y_pred_best == 0)
# False Positive: legit flagged as fraud
fp_mask = (y_test.values == 0) & (y_pred_best == 1)

# Pick one example from each (highest probability for TP/FP, lowest for FN)
tp_indices = np.where(tp_mask)[0]
fn_indices = np.where(fn_mask)[0]
fp_indices = np.where(fp_mask)[0]

# Sort by model probability to get the "most confident" examples
tp_idx = tp_indices[np.argmax(y_prob_best[tp_indices])] if len(tp_indices) > 0 else None
fn_idx = fn_indices[np.argmin(y_prob_best[fn_indices])] if len(fn_indices) > 0 else None
fp_idx = fp_indices[np.argmax(y_prob_best[fp_indices])] if len(fp_indices) > 0 else None

print("Selected transactions for local explanation:")
print(f"  True Positive  (idx={tp_idx}): P(fraud) = {y_prob_best[tp_idx]:.4f}" if tp_idx is not None else "  No TP found")
print(f"  False Negative (idx={fn_idx}): P(fraud) = {y_prob_best[fn_idx]:.4f}" if fn_idx is not None else "  No FN found")
print(f"  False Positive (idx={fp_idx}): P(fraud) = {y_prob_best[fp_idx]:.4f}" if fp_idx is not None else "  No FP found")

In [ ]:
# ---- Waterfall plots for each example transaction ----
# Create an Explanation object for cleaner waterfall/force plots
shap_explanation = shap.Explanation(
    values=shap_values_fraud,
    base_values=explainer.expected_value[1] if isinstance(explainer.expected_value, (list, np.ndarray)) else explainer.expected_value,
    data=X_test_shap.values,
    feature_names=X_test_shap.columns.tolist(),
)

cases = [
    ("True Positive (Fraud Correctly Flagged)", tp_idx),
    ("False Negative (Fraud Missed)", fn_idx),
    ("False Positive (Legit Flagged as Fraud)", fp_idx),
]

for title, idx in cases:
    if idx is None:
        print(f"\n{title}: No example available.")
        continue

    print(f"\n{'='*60}")
    print(f"{title}  |  P(fraud) = {y_prob_best[idx]:.4f}  |  True label = {y_test.values[idx]}")
    print(f"{'='*60}")

    plt.figure(figsize=(10, 6))
    shap.waterfall_plot(shap_explanation[idx], max_display=15, show=False)
    plt.title(f"Waterfall — {title}", fontsize=12, pad=20)
    plt.tight_layout()
    plt.show()

### 5.5 SHAP Analysis — Patterns & Business Takeaways

**Global Patterns Observed:**

1. **V14, V17, V12 dominate predictions** — consistent with the EDA correlation analysis in Part 1. These PCA components carry the strongest fraud signal, and SHAP confirms the model relies on them heavily. In a real-world setting (where original features are known), this would point to specific transaction attributes driving fraud risk.

2. **Direction of influence matters:**
   - Features with negative SHAP values when their raw value is low (blue dots pushing left) indicate that *unusually low values* of those features are associated with fraud. This is typical for PCA-transformed features that capture spending patterns or merchant categories.
   - Features with positive SHAP values when their raw value is high (red dots pushing right) indicate that *unusually high values* flag transactions as suspicious.

3. **Amount and Time are secondary signals** — they contribute, but far less than the PCA features. This aligns with our EDA finding: `Amount` has weak class separation and `Time` reflects daily cycles rather than fraud patterns.

4. **Feature interactions** — the dependence plots may reveal that certain features only become predictive *in combination* (e.g., V14 might only flag fraud when V17 is also abnormal). These interactions are exactly what tree-based models exploit but linear models miss.

**Local Patterns Observed:**

5. **True positives** show a clear "pile-up" of multiple features pushing strongly toward fraud — the model is confident because many signals align.

6. **False negatives** reveal fraud transactions that look "normal" on most features — only one or two features deviate slightly. These are the hardest cases and represent the fundamental precision-recall trade-off.

7. **False positives** show legitimate transactions with unusual patterns on a few key features — these could be high-value purchases, international transactions, or other legitimate-but-rare behaviors. In production, these are the cases where a human analyst adds the most value.

**Business Implications:**
- SHAP explanations can be attached to each flagged transaction as a **reason code** for the fraud review team -> "This transaction was flagged primarily because V14 and V17 showed anomalous values"
- The false negative analysis informs where the model's blind spots are -> targeted rule-based supplements can cover these gaps
- Feature importance rankings guide **feature engineering** priorities -> if we had access to the original (pre-PCA) features, we'd know exactly which data sources to invest in

**Next -> Part 6: Comprehensive Evaluation & Conclusion**

## Part 6 — Comprehensive Evaluation & Conclusion

This final section brings together all methods from Parts 3-4 into a unified evaluation framework. We compare supervised classifiers (across their best sampling strategies) and unsupervised anomaly detectors on the same untouched test set, using metrics that matter for imbalanced fraud detection.

**Roadmap:**
1. Unified comparison table (Precision / Recall / F1 / AUROC / AUPRC) across all methods
2. Overlaid ROC curves -> how well does each model rank transactions?
3. Overlaid PR curves -> how well does each model handle the rare-positive regime?
4. Best model's Confusion Matrix -> error breakdown at default threshold
5. Threshold analysis -> precision-recall trade-off as we move the decision boundary
6. Conclusion & Discussion -> best approach, limitations, future directions

### 6.1 Unified Comparison Table — All Methods

We collect the best (model, sampling) combo from each supervised model (Part 3) and both anomaly detection methods (Part 4) into a single table. All metrics are evaluated on the same held-out test set with the original imbalanced distribution.

In [ ]:
# ---- Retrain all best supervised models to get predictions ----
# We need predicted probabilities from each model for curve plotting

best_combos = {}
for model_name in ["Logistic Regression", "Random Forest", "XGBoost", "LightGBM"]:
    subset = results_df[results_df["Model"] == model_name]
    row = subset.loc[subset["AUPRC"].idxmax()]
    best_combos[model_name] = {
        "sampling": row["Sampling"],
        "params": row["Best Params"],
        "AUPRC": row["AUPRC"],
        "AUROC": row["AUROC"],
        "F1": row["F1"],
        "Precision": row["Precision"],
        "Recall": row["Recall"],
    }

# Retrain each to get y_prob on test set
model_probs = {}  # model_name -> y_prob array

for model_name, info in best_combos.items():
    samp = info["sampling"]
    params = info["params"]
    X_res, y_res = resampled_data[samp]

    if model_name == "Logistic Regression":
        cw = "balanced" if samp == "No Resampling" else None
        m = LogisticRegression(**params, max_iter=10000, class_weight=cw, random_state=SEED)
    elif model_name == "Random Forest":
        cw = "balanced" if samp == "No Resampling" else None
        m = RandomForestClassifier(**params, class_weight=cw, random_state=SEED, n_jobs=-1)
    elif model_name == "XGBoost":
        spw = (y_train == 0).sum() / (y_train == 1).sum() if samp == "No Resampling" else 1
        m = XGBClassifier(**params, scale_pos_weight=spw, eval_metric="aucpr",
                          use_label_encoder=False, random_state=SEED, n_jobs=-1)
    elif model_name == "LightGBM":
        is_unbal = True if samp == "No Resampling" else False
        m = LGBMClassifier(**params, is_unbalance=is_unbal, random_state=SEED, n_jobs=-1, verbose=-1)

    m.fit(X_res, y_res)
    model_probs[model_name] = m.predict_proba(X_test)[:, 1]

# Add anomaly detection scores (already computed in Part 4)
# Normalize anomaly scores to [0, 1] range for fair threshold comparison
from sklearn.preprocessing import MinMaxScaler

if_scores_norm = MinMaxScaler().fit_transform(if_scores_test.reshape(-1, 1)).ravel()
ae_scores_norm = MinMaxScaler().fit_transform(recon_error.reshape(-1, 1)).ravel()

model_probs["Isolation Forest"] = if_scores_norm
model_probs["Autoencoder"] = ae_scores_norm

print("All model probabilities / scores computed on test set.")
for name, probs in model_probs.items():
    print(f"  {name:>25s}  ->  shape={probs.shape}, range=[{probs.min():.4f}, {probs.max():.4f}]")

In [ ]:
# ---- Build unified comparison table ----
unified_rows = []

for model_name, probs in model_probs.items():
    # Determine threshold: 0.5 for supervised, best-F1 threshold for anomaly methods
    if model_name in ["Isolation Forest", "Autoencoder"]:
        prec_arr, rec_arr, thr_arr = precision_recall_curve(y_test, probs)
        f1_arr = 2 * (prec_arr * rec_arr) / (prec_arr + rec_arr + 1e-8)
        best_thr_idx = np.argmax(f1_arr)
        threshold = thr_arr[best_thr_idx]
        sampling_label = "Semi-supervised" if model_name == "Autoencoder" else "Unsupervised"
    else:
        threshold = 0.5
        sampling_label = best_combos[model_name]["sampling"]

    y_pred = (probs >= threshold).astype(int)

    unified_rows.append({
        "Model": model_name,
        "Sampling": sampling_label,
        "Threshold": threshold,
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "F2": fbeta_score(y_test, y_pred, beta=2),
        "AUROC": roc_auc_score(y_test, probs),
        "AUPRC": average_precision_score(y_test, probs),
    })

unified_df = pd.DataFrame(unified_rows).sort_values("AUPRC", ascending=False).reset_index(drop=True)

unified_df.style.format({
    "Threshold": "{:.4f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1": "{:.4f}",
    "F2": "{:.4f}",
    "AUROC": "{:.4f}",
    "AUPRC": "{:.4f}",
}).background_gradient(subset=["AUPRC", "AUROC", "F1", "F2"], cmap="YlGn")

### 6.2 ROC Curves — All Methods Overlaid

ROC curves plot True Positive Rate vs. False Positive Rate across all thresholds. A model closer to the top-left corner has better overall ranking quality. The diagonal represents a random classifier.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

colors_roc = {
    "Logistic Regression": "#e74c3c",
    "Random Forest":       "#3498db",
    "XGBoost":             "#2ecc71",
    "LightGBM":            "#9b59b6",
    "Isolation Forest":    "#e67e22",
    "Autoencoder":         "#1abc9c",
}

for model_name, probs in model_probs.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    auroc = roc_auc_score(y_test, probs)
    linestyle = "--" if model_name in ["Isolation Forest", "Autoencoder"] else "-"
    ax.plot(fpr, tpr, label=f"{model_name} (AUROC={auroc:.4f})",
            color=colors_roc[model_name], linestyle=linestyle, linewidth=1.8)

ax.plot([0, 1], [0, 1], "k--", alpha=0.3, label="Random (AUROC=0.5)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curves — All Methods", fontsize=14)
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim([-0.01, 1.0])
ax.set_ylim([0.0, 1.01])
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### 6.3 Precision-Recall Curves — All Methods Overlaid

For highly imbalanced tasks, PR curves are more informative than ROC curves. A model closer to the top-right corner achieves high precision at high recall — the ideal for fraud detection. The horizontal baseline is the fraud prevalence (~0.17%).

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

for model_name, probs in model_probs.items():
    prec, rec, _ = precision_recall_curve(y_test, probs)
    auprc = average_precision_score(y_test, probs)
    linestyle = "--" if model_name in ["Isolation Forest", "Autoencoder"] else "-"
    ax.plot(rec, prec, label=f"{model_name} (AUPRC={auprc:.4f})",
            color=colors_roc[model_name], linestyle=linestyle, linewidth=1.8)

baseline = y_test.mean()
ax.axhline(y=baseline, color="k", linestyle="--", alpha=0.3, label=f"Baseline (prevalence={baseline:.4f})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curves — All Methods", fontsize=14)
ax.legend(loc="upper right", fontsize=9)
ax.set_xlim([0.0, 1.01])
ax.set_ylim([0.0, 1.01])
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()

### 6.4 Best Model — Confusion Matrix

We visualize the confusion matrix of the overall best model (by AUPRC) at its default threshold. This breaks down exactly how many transactions are correctly classified, missed (false negatives), and falsely flagged (false positives).

In [ ]:
# ---- Identify the overall best model by AUPRC ----
best_overall = unified_df.iloc[0]
best_name = best_overall["Model"]
best_threshold = best_overall["Threshold"]

print(f"Best model: {best_name} (AUPRC = {best_overall['AUPRC']:.4f})")
print(f"Threshold:  {best_threshold:.4f}\n")

y_pred_final = (model_probs[best_name] >= best_threshold).astype(int)
cm = confusion_matrix(y_test, y_pred_final)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", linewidths=0.5,
            xticklabels=["Legit", "Fraud"], yticklabels=["Legit", "Fraud"], ax=ax,
            annot_kws={"fontsize": 14})
ax.set_xlabel("Predicted", fontsize=12)
ax.set_ylabel("Actual", fontsize=12)
ax.set_title(f"Confusion Matrix — {best_name} (threshold={best_threshold:.2f})", fontsize=14)
plt.tight_layout()
plt.show()

# Print classification report
print(f"\nClassification Report — {best_name}:\n")
print(classification_report(y_test, y_pred_final, target_names=["Legit", "Fraud"], digits=4))

### 6.5 Threshold Analysis — Precision-Recall Trade-Off

In production fraud detection, the decision threshold is a **business decision**, not just a modeling choice. Lowering the threshold catches more fraud (higher recall) but increases false alarms (lower precision), which means more manual reviews. Raising the threshold reduces false alarms but lets more fraud slip through.

This analysis shows how precision, recall, and F1 shift as we sweep the threshold from 0 to 1, and helps identify the operating point that best balances detection and review cost.

In [ ]:
# ---- Threshold sweep for the best model ----
best_probs = model_probs[best_name]

thresholds = np.linspace(0.01, 0.99, 200)
precision_arr = []
recall_arr = []
f1_arr = []
f2_arr = []
n_flagged_arr = []

for t in thresholds:
    preds = (best_probs >= t).astype(int)
    n_flagged = preds.sum()
    if n_flagged == 0 or (preds == 1).sum() == 0:
        precision_arr.append(0)
        recall_arr.append(0)
        f1_arr.append(0)
        f2_arr.append(0)
    else:
        precision_arr.append(precision_score(y_test, preds, zero_division=0))
        recall_arr.append(recall_score(y_test, preds, zero_division=0))
        f1_arr.append(f1_score(y_test, preds, zero_division=0))
        f2_arr.append(fbeta_score(y_test, preds, beta=2, zero_division=0))
    n_flagged_arr.append(n_flagged)

precision_arr = np.array(precision_arr)
recall_arr = np.array(recall_arr)
f1_arr = np.array(f1_arr)
f2_arr = np.array(f2_arr)
n_flagged_arr = np.array(n_flagged_arr)

# Best F1 and F2 thresholds
best_f1_idx = np.argmax(f1_arr)
best_f1_threshold = thresholds[best_f1_idx]
best_f2_idx = np.argmax(f2_arr)
best_f2_threshold = thresholds[best_f2_idx]

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ---- Left: Precision, Recall, F1, F2 vs Threshold ----
ax = axes[0]
ax.plot(thresholds, precision_arr, label="Precision", color="#3498db", linewidth=1.8)
ax.plot(thresholds, recall_arr, label="Recall", color="#e74c3c", linewidth=1.8)
ax.plot(thresholds, f1_arr, label="F1 Score", color="#2ecc71", linewidth=1.8)
ax.plot(thresholds, f2_arr, label="F2 Score", color="#e67e22", linewidth=1.8, linestyle="--")
ax.axvline(x=best_f1_threshold, color="#2ecc71", linestyle="--", alpha=0.5,
           label=f"Best F1 threshold = {best_f1_threshold:.3f}")
ax.axvline(x=best_f2_threshold, color="#e67e22", linestyle=":", alpha=0.5,
           label=f"Best F2 threshold = {best_f2_threshold:.3f}")
ax.axvline(x=0.5, color="black", linestyle=":", alpha=0.4, label="Default threshold (0.5)")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Score")
ax.set_title(f"Threshold Analysis — {best_name}", fontsize=13)
ax.legend(fontsize=8, loc="center left")
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.05])
ax.grid(True, alpha=0.2)

# ---- Right: Number of flagged transactions vs threshold ----
ax = axes[1]
ax.plot(thresholds, n_flagged_arr, color="#9b59b6", linewidth=1.8)
ax.axvline(x=best_f1_threshold, color="#2ecc71", linestyle="--", alpha=0.5, label="Best F1 threshold")
ax.axvline(x=best_f2_threshold, color="#e67e22", linestyle=":", alpha=0.5, label="Best F2 threshold")
ax.axvline(x=0.5, color="black", linestyle=":", alpha=0.4, label="Default (0.5)")
ax.axhline(y=y_test.sum(), color="#e74c3c", linestyle="--", alpha=0.5,
           label=f"Actual fraud count ({y_test.sum()})")
ax.set_xlabel("Decision Threshold")
ax.set_ylabel("Number of Flagged Transactions")
ax.set_title("Review Volume vs. Threshold", fontsize=13)
ax.legend(fontsize=9)
ax.set_xlim([0, 1])
ax.grid(True, alpha=0.2)

plt.tight_layout()
plt.show()

print(f"\nThreshold Analysis Summary for {best_name}:")
print(f"  Default (0.5)       ->  Prec={precision_arr[np.argmin(np.abs(thresholds - 0.5))]:.4f}  "
      f"Recall={recall_arr[np.argmin(np.abs(thresholds - 0.5))]:.4f}  "
      f"F1={f1_arr[np.argmin(np.abs(thresholds - 0.5))]:.4f}  "
      f"F2={f2_arr[np.argmin(np.abs(thresholds - 0.5))]:.4f}  "
      f"Flagged={n_flagged_arr[np.argmin(np.abs(thresholds - 0.5))]:,}")
print(f"  Best F1 ({best_f1_threshold:.3f})   ->  Prec={precision_arr[best_f1_idx]:.4f}  "
      f"Recall={recall_arr[best_f1_idx]:.4f}  "
      f"F1={f1_arr[best_f1_idx]:.4f}  "
      f"F2={f2_arr[best_f2_idx]:.4f}  "
      f"Flagged={n_flagged_arr[best_f1_idx]:,}")
print(f"  Best F2 ({best_f2_threshold:.3f})   ->  Prec={precision_arr[best_f2_idx]:.4f}  "
      f"Recall={recall_arr[best_f2_idx]:.4f}  "
      f"F1={f1_arr[best_f2_idx]:.4f}  "
      f"F2={f2_arr[best_f2_idx]:.4f}  "
      f"Flagged={n_flagged_arr[best_f2_idx]:,}")

In [ ]:
# ---- Business scenario comparison: different threshold strategies ----
scenarios = {
    "High Recall (catch most fraud)": 0.2,
    "Balanced — Best F2 (recall-weighted)": best_f2_threshold,
    "Balanced — Best F1 (equal weight)": best_f1_threshold,
    "Default (0.5)": 0.5,
    "High Precision (minimize false alarms)": 0.8,
}

scenario_rows = []
for label, t in scenarios.items():
    preds = (best_probs >= t).astype(int)
    n_flag = preds.sum()
    tp = ((preds == 1) & (y_test.values == 1)).sum()
    fp = ((preds == 1) & (y_test.values == 0)).sum()
    fn = ((preds == 0) & (y_test.values == 1)).sum()
    scenario_rows.append({
        "Strategy": label,
        "Threshold": t,
        "Fraud Caught (TP)": tp,
        "Fraud Missed (FN)": fn,
        "False Alarms (FP)": fp,
        "Total Flagged": n_flag,
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1": f1_score(y_test, preds, zero_division=0),
        "F2": fbeta_score(y_test, preds, beta=2, zero_division=0),
    })

scenario_df = pd.DataFrame(scenario_rows)
scenario_df.style.format({
    "Threshold": "{:.3f}",
    "Precision": "{:.4f}",
    "Recall": "{:.4f}",
    "F1": "{:.4f}",
    "F2": "{:.4f}",
}).background_gradient(subset=["F1", "F2"], cmap="YlGn")

#### Threshold Recommendation

The threshold analysis reveals a clear trade-off between fraud detection rate and review volume:
- A **lower threshold** (~0.2) maximizes recall -> suitable for high-risk environments where missing fraud is very costly
- The **best-F2 threshold** prioritizes recall over precision (recall is weighted 2x) -> recommended for fraud detection where the cost of a missed fraud far exceeds the cost of a false alarm
- The **best-F1 threshold** balances precision and recall equally -> suitable when review capacity and fraud loss are roughly symmetric
- A **higher threshold** (~0.8) minimizes false alarms -> suitable when review capacity is severely limited

**Why F2 matters here:** In fraud detection, a missed fraudulent transaction (false negative) typically costs the bank the full transaction amount plus customer trust, while a false alarm (false positive) only costs the time to review it. F2 captures this asymmetry by weighting recall twice as heavily as precision, making it a more operationally relevant metric than F1 for this domain. The gap between the best-F1 and best-F2 thresholds quantifies exactly how much additional fraud can be caught by accepting a moderate increase in false alarms.

In [ ]:
# ---- Final summary print ----
print("=" * 70)
print("PROJECT SUMMARY — Credit Card Fraud Detection")
print("=" * 70)
print(f"\nDataset: 284,807 transactions | 492 fraud ({492/284807:.4%})")
print(f"Test set: {X_test.shape[0]:,} transactions | {y_test.sum()} fraud")
print(f"\nBest supervised model:  {unified_df.iloc[0]['Model']}")
print(f"  Sampling strategy:    {unified_df.iloc[0]['Sampling']}")
print(f"  AUPRC = {unified_df.iloc[0]['AUPRC']:.4f}")
print(f"  AUROC = {unified_df.iloc[0]['AUROC']:.4f}")
print(f"  F1    = {unified_df.iloc[0]['F1']:.4f}  |  F2 = {unified_df.iloc[0]['F2']:.4f}")
print(f"\nMethods explored:")
print(f"  Supervised:   Logistic Regression, Random Forest, XGBoost, LightGBM")
print(f"  Unsupervised: Isolation Forest, Autoencoder (PyTorch)")
print(f"  Sampling:     No Resampling, ROS, SMOTE, ADASYN, SMOTE + Tomek")
print(f"  Explainability: SHAP (global + local)")
print(f"\nKey insight: Threshold tuning is as important as model selection —")
print(f"  the best F1 threshold ({best_f1_threshold:.3f}) differs from the default (0.5),")
print(f"  and the right choice depends on business cost trade-offs.")
print("=" * 70)

Thanks! Good luck!